# CS 171 / EE 142 Problem Set 32
# Due Friday, May 8, 2026 @ 11:59pm

## Read *all* cells carefully and answer all parts (both text and missing code)

### Enter your information below:

<div style="color: #000000;background-color: #EEEEFF">
    Your Name (submitter): <br>
Your student ID (submitter):
    
<b>By submitting this notebook, I assert that the work below is my own work, completed for this course.  Except where explicitly cited, none of the portions of this notebook are duplicated from anyone else's work or my own previous work.</b>
</div>


<div class="alert alert-success">
    <font size=+2>Total Problem Set Grading</font> (to be completed by grader)<br>
    Total Points: /20<br>
    Late Days Used on this Assignment: <br>
    Total Late Days Used: <br>
</div>

<div style="color: #000000;background-color: #FFEEDD">
<h2>Overview</h2>

In this problem set, we will be using the "fashion MNIST" classification dataset (https://github.com/zalandoresearch/fashion-mnist), slightly modified.  In this modified version, there are four classes, each corresponding to a different item of clothing (that might be sold on a website, for instance).  The features are the pixels of an image of the item.  The goal is to be able to classify the type of clothing from the image.
- The data have been "z-normalized:" each feature has been scaled and shifted so that "0" is the average value and the magnitude is the number of standard deviations from the mean value.  We will learn more about this later.  For now, just note that the average value of each feature is 0.
- The data have been split into training and testing data.
- Only 4 classes are used (the full dataset has 10 classes), and only half of the training data is used (to make the problem set faster).

The code below loads in the variables:
- `trainX` is the standard training data matrix.
- `testX` is the corresponding data matrix for the testing data.
- `trainY` is the target for the classification problems: 0, 1, 2, or 3
- `testY` is the same for the testing set
- `ycat` is the names of the classes (not necessary).

Note that throughout this assignment, you will be performing calculations or transformations that will be needed in later parts of the assignment.  Writing general functions will save time and hassle later.

The first part of the assignment will only deal with examples with label 0 or label 1.  The second part will deal with the whole dataset.
</div>

In [ ]:
# no other libraries are to be imported!
import numpy as np
import matplotlib.pyplot as plt

vars = np.load('fashionmod.npz',allow_pickle=True)
for k in vars.keys():
    exec(f"{k} = vars['{k}']")

<div style="color: #000000;background-color: #FFEEFF">
    <font size=+2>Part I: Binary Classification</font>
</div>

The function below returns a learned *function*. It is called with:
* a dataset (X and Y),
* the regularization parameter (C),
* the kernel type (a string: 'linear','poly', or 'rbf'), and
* any parameters that kernel needs.
  
For instance, `f=learnsvm(X,Y,10,'poly',c=1,d=2)` learns a support vector machine with a second-degree polynomial kernel

Given testing data, `f(testX)` will return a vector of the discriminant values, one for each row of `testX`

In [ ]:
def learnsvm(X,Y,C,kerneltype,**kernelparams):
    if ((Y!=+1) & (Y!=-1)).sum()>0:
        raise Exception('Only binary SVM with classes +1/-1 allowed')
    if kerneltype not in ['linear','poly','rbf']:
        raise Exception('kernel must be one of "linear", "poly", or "rbf"')
    if kerneltype=='poly':
        if 'c' not in kernelparams or 'd' not in kernelparams:
            raise Exception('poly kernel must supply parameters c and d')
    if kerneltype=='rbf':
        if 'gamma' not in kernelparams:
            raise Exception('gamma kernel must supply parameter gamma')
            
    from sklearn.svm import SVC # you may not import anything else from sklearn (or any other library)
    if 'c' in kernelparams:
        kernelparams['coef0']=kernelparams['c']
        del kernelparams['c']
    if 'd' in kernelparams:
        kernelparams['degree']=kernelparams['d']
        del kernelparams['d']
    if 'gamma' not in kernelparams:
        kernelparams['gamma'] = 1.0
    kernelparams['kernel'] = kerneltype
    model = SVC(C=C,**kernelparams,tol=1e-3,shrinking=False)
    model.fit(X,Y.astype(int))
    return model.decision_function

<div style="color: #000000;background-color: #FFFFEE">
    <font size=+2>Question 1:</font> <font size=+1>(2 points)</font>
    
Use only class 0 and class 1.  You will need to convert the provided training and testing data to only contain examples from class 0 and class 1 (and remap these to class -1 and class +1).

Report the testing accuracy (fraction of testing examples properly classified) using a linear support vector machine with C=0.01.  Use the code in the cell above to learn the SVM.
</div>
<div class="alert alert-success">
    <font size=+1>Grading</font> (to be completed by grader)<br>
    Score: /2<br>
</div>

In [ ]:
%%time
mask_train = (trainY == 0) | (trainY == 1)
mask_test = (testY == 0) | (testY == 1)

X_bin_train = trainX[mask_train]
Y_bin_train_mapped = np.where(trainY[mask_train] == 0, -1, 1)
X_bin_test = testX[mask_test]
Y_bin_test_mapped = np.where(testY[mask_test] == 0, -1, 1)

f_q1 = learnsvm(X_bin_train, Y_bin_train_mapped, 0.01, 'linear')
preds = np.sign(f_q1(X_bin_test))
print(f"test accuracy: {np.mean(preds == Y_bin_test_mapped):.4f}")

<div style="color: #000000;background-color: #FFFFEE">
    <font size=+2>Question 2:</font> <font size=+1>(4 points)</font>
    
Again, use only class 0 and class 1.

Use hold-out validation to select the kernel and its parameters.  You should use 50% of the training data as validation data.  (The data have already been shuffled, so just use the first half for parameter fitting and the second half for validation.)

You should use validation to select between a linear kernel and a polynomial kernel.  The polynomial kernel can be either of degree 2 or degree 3.  The polynomial kernel's c parameter can be either 1 or 10.  For C, select among 5 evenly distributed points between 1e-5 and 1e-3 *in log space*.  The function `np.logspace(-5,-3,5)` will provide this list.  All combinations of hyper-parameters will need to be tried.  The final classifier should be trained on all of the training data using these best hyper-parameters.

Report the chosen kernel, kernel parameters, and C, as well as the resulting testing accuracy.
</div>
<div class="alert alert-success">
    <font size=+1>Grading</font> (to be completed by grader)<br>
    Score: /4<br>
</div>

In [ ]:
%%time
half = len(X_bin_train) // 2
X_fit, Y_fit = X_bin_train[:half], Y_bin_train_mapped[:half]
X_val, Y_val = X_bin_train[half:], Y_bin_train_mapped[half:]

C_values = np.logspace(-5, -3, 5)
best_val_acc = -1
best_params = None

for C in C_values:
    f = learnsvm(X_fit, Y_fit, C, 'linear')
    acc = np.mean(np.sign(f(X_val)) == Y_val)
    if acc > best_val_acc:
        best_val_acc = acc
        best_params = {'kernel': 'linear', 'C': C}

for d in [2, 3]:
    for c in [1, 10]:
        for C in C_values:
            f = learnsvm(X_fit, Y_fit, C, 'poly', c=c, d=d)
            acc = np.mean(np.sign(f(X_val)) == Y_val)
            if acc > best_val_acc:
                best_val_acc = acc
                best_params = {'kernel': 'poly', 'C': C, 'c': c, 'd': d}

print(f"best val acc: {best_val_acc:.4f}, params: {best_params}")

if best_params['kernel'] == 'linear':
    f_best_q2 = learnsvm(X_bin_train, Y_bin_train_mapped, best_params['C'], 'linear')
else:
    f_best_q2 = learnsvm(X_bin_train, Y_bin_train_mapped, best_params['C'], 'poly',
                         c=best_params['c'], d=best_params['d'])

preds = np.sign(f_best_q2(X_bin_test))
print(f"test accuracy: {np.mean(preds == Y_bin_test_mapped):.4f}")

<div style="color: #000000;background-color: #FFEEFF">
    <font size=+2>Part II: Multi-Class Classification</font>
</div>

<div style="color: #000000;background-color: #FFFFEE">
    <font size=+2>Question 3:</font> <font size=+1>(8 points)</font>

Now we will use all of the classes.  We will generate a 4-way classifier from binary classifiers using a method called "1-v-1."  For each pair of classes, train a classifer to separate those classes (using the hold-out validation method from question 2).  Question 2 essentially did this for class 0 versus class 1, for instance.  For 4 classes, this is 6 different classifiers.  Do not "hard code" this.  Make a learning algorithm that will work for any number of classes.

Then, to predict, each classifer "votes" on the final class.  (A classifier can only vote for one of the two classes it was trained on.)  The final prediction is the class that receives the most votes.

Implement this 1-v-1 classifier and report the accuracy on the entire testing set (the fraction of the testing examples it get correct).
</div>
   <div class="alert alert-success">
    <font size=+1>Grading</font> (to be completed by grader)<br>
    Score: /8<br>
</div>

In [ ]:
%%time
def train_binary(X, Y):
    half = len(X) // 2
    X_fit, Y_fit = X[:half], Y[:half]
    X_val, Y_val = X[half:], Y[half:]

    C_values = np.logspace(-5, -3, 5)
    best_acc = -1
    best_params = None

    for C in C_values:
        f = learnsvm(X_fit, Y_fit, C, 'linear')
        acc = np.mean(np.sign(f(X_val)) == Y_val)
        if acc > best_acc:
            best_acc = acc
            best_params = {'kernel': 'linear', 'C': C}

    for d in [2, 3]:
        for c in [1, 10]:
            for C in C_values:
                f = learnsvm(X_fit, Y_fit, C, 'poly', c=c, d=d)
                acc = np.mean(np.sign(f(X_val)) == Y_val)
                if acc > best_acc:
                    best_acc = acc
                    best_params = {'kernel': 'poly', 'C': C, 'c': c, 'd': d}

    if best_params['kernel'] == 'linear':
        return learnsvm(X, Y, best_params['C'], 'linear'), best_params
    else:
        return learnsvm(X, Y, best_params['C'], 'poly',
                        c=best_params['c'], d=best_params['d']), best_params


classes = np.unique(trainY)
n_classes = len(classes)
classifiers = {}

for i in range(n_classes):
    for j in range(i + 1, n_classes):
        ci, cj = classes[i], classes[j]
        mask = (trainY == ci) | (trainY == cj)
        X_pair = trainX[mask]
        Y_pair = np.where(trainY[mask] == ci, -1, 1)
        print(f"training {ci} vs {cj}...")
        f, params = train_binary(X_pair, Y_pair)
        classifiers[(ci, cj)] = (f, ci, cj)
        print(f"  {params}")


def predict_ovo(X):
    class_to_idx = {c: i for i, c in enumerate(classes)}
    votes = np.zeros((len(X), n_classes), dtype=int)
    for (ci, cj), (f, _, _) in classifiers.items():
        disc = f(X)
        votes[disc < 0, class_to_idx[ci]] += 1
        votes[disc >= 0, class_to_idx[cj]] += 1
    return classes[np.argmax(votes, axis=1)]


test_preds = predict_ovo(testX)
print(f"\n1v1 test accuracy: {np.mean(test_preds == testY):.4f}")

<div style="color: #000000;background-color: #FFFFEE">
    <font size=+2>Question 4:</font> <font size=+1>(4 points)</font>
<p>    Using the same learned multi-class classifier from question 3, calculate and show the confusion matrix for the testing data.</p>
<p>The confusion matrix is a C-by-C matrix (if C is the number of classes).  The element in row i, column j is the number of
examples whose true label is i which are classified as j.  Therefore, the diagonal elements are counts of the examples that are correctly
classified and the off-diagonal elements count different types of misclassifications.</p>
</div>
   <div class="alert alert-success">
    <font size=+1>Grading</font> (to be completed by grader)<br>
    Score: /4<br>
</div>

In [ ]:
conf_matrix = np.zeros((n_classes, n_classes), dtype=int)
for true_lbl, pred_lbl in zip(testY, test_preds):
    i = np.where(classes == true_lbl)[0][0]
    j = np.where(classes == pred_lbl)[0][0]
    conf_matrix[i, j] += 1

print(conf_matrix)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(conf_matrix, cmap='Blues')
ax.set_xticks(range(n_classes))
ax.set_yticks(range(n_classes))
ax.set_xticklabels([f'Pred {c}' for c in classes])
ax.set_yticklabels([f'True {c}' for c in classes])
for i in range(n_classes):
    for j in range(n_classes):
        ax.text(j, i, str(conf_matrix[i, j]), ha='center', va='center',
                color='white' if conf_matrix[i, j] > conf_matrix.max() / 2 else 'black')
plt.colorbar(im)
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

<div style="color: #000000;background-color: #FFFFEE">
    <font size=+2>Question 5:</font> <font size=+1>(2 points)</font>
    What can you say about these classes from the confusion matrix?  Looking at the labels (<tt>ycats</tt>), does this make sense?
</div>
   <div class="alert alert-success">
    <font size=+1>Grading</font> (to be completed by grader)<br>
    Score: /2<br>
</div>

## Your answer here

Labels 0 and 1 are sometimes confused, as are labels 2 and 3.  However, 0/1 is almost never confused with 2/3.  This makes sense because labels 0 and 1 correspond to two different types of shoes (sneakers and ankle boots) and labels 2 and 3 both refer to garments for the torso (top and shirt).